# 💬 Текст. Как превратить твит в числа и угадать его настроение

> Сначала: **Файл → Сохранить копию на Диске**.

У нас 10 000 русских твитов, у каждого метка: **позитив** или **негатив**. Модель умеет работать только с **числами**, поэтому сегодня:
1. **Токены** — режем твит на слова.
2. **Словарь** — нумеруем все слова.
3. **One-hot** — слово → вектор из нулей и одной единицы.
4. **Мешок слов (Bag of Words)** — твит → сколько раз встретилось каждое слово.
5. **TF-IDF** — приглушаем слова, которые есть везде.
6. **Модель** угадывает настроение твита.

Каждое задание: замените `None` на свой код → **Shift + Enter** → запустите ячейку 🧪 под ним.

In [ ]:
#@title ⚙️ Запустите первой: библиотеки, твиты и проверки { display-mode: "form" }
# Внутри только подготовка. Читать не обязательно :)
import math, os, re, urllib.request
import numpy as np
import pandas as pd

if not os.path.exists("tweets.csv"):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/Scr1ptG/ml_for_3p1/main/06-text/data/tweets.csv", "tweets.csv")


def check(tests, n):
    for i, (name, cond, hint) in enumerate(tests, 1):
        try:
            passed = bool(cond())
        except Exception:
            passed = False
        if not passed:
            print(f"❌ Тест {i} из {len(tests)}: {name}\n   💡 {hint}")
            return False
        print(f"✅ Тест {i}: {name}")
    print(f"\n🎉 Задание {n} выполнено!")
    return True


print("✅ Готово. Файл tweets.csv скачан:", os.path.exists("tweets.csv"))

---
# Часть 1. Смотрим на твиты

In [ ]:
df = pd.read_csv("tweets.csv")
df.head(10)

### ✏️ Задание 1. Сколько всего твитов `n` и сколько из них позитивных `n_pos`?

In [ ]:
n = None
n_pos = None

In [ ]:
#@title 🧪 Проверка задания 1 { display-mode: "form" }
if check([
    ('всего твитов', lambda: n == 10000, 'Сколько строк в таблице?'),
    ('позитивных', lambda: n_pos == 5000, 'Сравните столбец label с нужным словом и посчитайте совпадения'),
], 1):
    print(f"Всего {n}, позитивных {n_pos}, негативных {n - n_pos}")
    print("Классы поровну: если угадывать наугад, будет 50% правильных.")

<details><summary>💡 Подсказка</summary>

Длина таблицы — встроенная функция Python. Сравнение столбца со строкой даёт True/False, а True считаются как единицы.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
n = ___(df)
n_pos = (df["label"] == ___).___()
```
</details>

> ℹ️ Метки в этом наборе ставили **по смайликам**: `:)` — позитив, `:(` — негатив. Смайлики мы убрали, иначе было бы слишком просто. Модели придётся угадывать настроение **по словам**.

---
# Часть 2. Токены: режем текст на слова

**Токен** — кусочек текста, с которым работает модель. У нас токен = слово.

Первая идея — `split()`: резать по пробелам.

In [ ]:
t = "Отличный был денёк! Отличного вам вечера, друзья!"
print(t.split())

Плохо: `денёк!` и `денёк` — разные «слова», `Отличный` и `отличный` — тоже. Поэтому:
1. переводим в **нижний регистр**;
2. берём только **буквы**: `re.findall(r"[а-яё]+", text)` находит все куски из русских букв.

### ✏️ Задание 2. Напишите функцию `tokenize(text)`: нижний регистр + только русские слова

**Что должно получиться:** `['отличный', 'был', 'денёк', 'отличного', 'вам', 'вечера', 'друзья']`

In [ ]:
def tokenize(text):
    return None


In [ ]:
#@title 🧪 Проверка задания 2 { display-mode: "form" }
if check([
    ('возвращает список слов', lambda: tokenize('Привет, мир!') == ['привет', 'мир'], 'Нужен re.findall с шаблоном русских букв'),
    ('нижний регистр', lambda: tokenize('ДЕНЁК') == ['денёк'], 'Шаблон ищет только маленькие буквы — сначала переведите текст в нижний регистр'),
], 2):
    print(tokenize(t))
    print(tokenize(df["text"][0]))

<details><summary>💡 Подсказка</summary>

У строки есть метод для нижнего регистра. `re.findall(шаблон, строка)` возвращает список найденных кусков.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
return re.findall(___, text.___())
```
</details>

---
# Часть 3. Словарь: нумеруем слова

Начнём с маленького примера — 4 «твита»:

In [ ]:
corpus = [
    "Кот спит. Кот доволен",
    "Пёс спит",
    "не плохо, а хорошо",
    "не хорошо, а плохо",
]
all_words = []
for text in corpus:
    all_words += tokenize(text)
print(all_words)

### ✏️ Задание 3. Соберите словарь `vocab`: все **разные** слова корпуса, по алфавиту

In [ ]:
vocab = None

In [ ]:
#@title 🧪 Проверка задания 3 { display-mode: "form" }
if check([
    ('только разные слова', lambda: vocab is not None and len(vocab) == len(set(vocab)) == 8, 'Какой тип в Python хранит элементы без повторов?'),
    ('по алфавиту', lambda: vocab == sorted(vocab), 'Отсортируйте'),
], 3):
    print("Словарь:", vocab)
    print("Размер словаря:", len(vocab))

<details><summary>💡 Подсказка</summary>

Повторы убирает `set`, упорядочивает — `sorted`.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
vocab = ___(___(all_words))
```
</details>

### ✏️ Задание 4. Сделайте `word2idx`: словарь «слово → его номер в vocab»

In [ ]:
word2idx = None

In [ ]:
#@title 🧪 Проверка задания 4 { display-mode: "form" }
if check([
    ('все слова есть', lambda: word2idx is not None and len(word2idx) == len(vocab), 'В словаре должно быть столько же слов, сколько в vocab'),
    ('номер = место в vocab', lambda: all(vocab[i] == w for w, i in word2idx.items()), 'Номер слова — его позиция в списке vocab'),
], 4):
    print(word2idx)
    print("Номер слова «спит»:", word2idx["спит"])

<details><summary>💡 Подсказка</summary>

`enumerate(vocab)` выдаёт пары `(номер, слово)`. Соберите из них dict comprehension.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
word2idx = {___: ___ for i, w in enumerate(vocab)}
```
</details>

---
# Часть 4. One-hot: слово → вектор

Слово превращаем в вектор длины «размер словаря»: везде **0**, и только на месте этого слова **1**.

| | а | доволен | кот | не | пёс | плохо | спит | хорошо |
|---|---|---|---|---|---|---|---|---|
| **кот** | 0 | 0 | **1** | 0 | 0 | 0 | 0 | 0 |
| **спит** | 0 | 0 | 0 | 0 | 0 | 0 | **1** | 0 |

### ✏️ Задание 5. Функция `one_hot(word)`

In [ ]:
def one_hot(word):
    v = np.zeros(len(vocab), dtype=int)
    # ваш код
    return v

In [ ]:
#@title 🧪 Проверка задания 5 { display-mode: "form" }
if check([
    ('одна единица', lambda: one_hot('кот').sum() == 1, 'В векторе должна появиться ровно одна 1'),
    ('на нужном месте', lambda: one_hot('кот')[word2idx['кот']] == 1 and one_hot('спит')[word2idx['спит']] == 1, 'Место единицы — номер слова из word2idx'),
], 5):
    for w in ["кот", "спит", "хорошо"]:
        print(f"{w:>7}", one_hot(w))

<details><summary>💡 Подсказка</summary>

Узнайте номер слова через `word2idx` и запишите 1 в эту ячейку вектора.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
v[___] = 1
```
</details>

### ✏️ Задание 6. Сколько разных слов во всех твитах? Переменная `V`

In [ ]:
real_vocab = set()
for text in df["text"]:
    real_vocab.update(tokenize(text))

V = None

In [ ]:
#@title 🧪 Проверка задания 6 { display-mode: "form" }
if check([
    ('размер словаря', lambda: V == len(real_vocab) and V > 20000, 'Сколько элементов в множестве real_vocab?'),
], 6):
    print(f"Разных слов: {V}")
    print(f"Значит, one-hot вектор каждого слова — {V} чисел, и из них только ОДНА единица!")

<details><summary>💡 Подсказка</summary>

Та же функция, что считает длину списка.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
V = ___
```
</details>

Вывод: one-hot — **длинный** и почти весь из нулей. Ещё он ничего не знает о смысле: «хорошо» и «отлично» так же далеки, как «хорошо» и «кот». (Это решают **эмбеддинги**, например word2vec.)

---
# Часть 5. Мешок слов (Bag of Words)

Твит = **сумма one-hot векторов** его слов. Получаем: сколько раз каждое слово встретилось в твите.

«Кот спит. Кот доволен» → `кот` 2 раза, `спит` 1, `доволен` 1:

| а | доволен | кот | не | пёс | плохо | спит | хорошо |
|---|---|---|---|---|---|---|---|
| 0 | 1 | **2** | 0 | 0 | 0 | 1 | 0 |

Почему «мешок»: слова высыпали в мешок и посчитали. **Порядок слов потерялся.**

### ✏️ Задание 7. Функция `bow(text)`: мешок слов для одного текста

In [ ]:
def bow(text):
    v = np.zeros(len(vocab), dtype=int)
    for w in tokenize(text):
        if w in word2idx:          # незнакомые слова пропускаем
            pass               # ваш код
    return v

In [ ]:
#@title 🧪 Проверка задания 7 { display-mode: "form" }
if check([
    ('кот посчитан дважды', lambda: bow('Кот спит. Кот доволен')[word2idx['кот']] == 2, 'Слово может встретиться несколько раз — не ставьте, а прибавляйте'),
    ('всего 4 слова', lambda: bow('Кот спит. Кот доволен').sum() == 4, 'Сумма вектора должна равняться числу слов'),
    ('незнакомые слова не ломают', lambda: bow('кот и собака').sum() == 1, 'Проверка if w in word2idx уже есть'),
], 7):
    print(vocab)
    for text in corpus:
        print(bow(text), "←", text)

<details><summary>💡 Подсказка</summary>

Почти как `one_hot`, только слов много и каждое добавляет единицу к своей ячейке.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
v[___] ___ 1
```
</details>

### ✏️ Задание 8. Одинаковы ли мешки у «не плохо, а хорошо» и «не хорошо, а плохо»? Переменная `same`

In [ ]:
same = None

In [ ]:
#@title 🧪 Проверка задания 8 { display-mode: "form" }
if check([
    ('сравнили', lambda: same == True, 'Сравните два вектора целиком: в numpy есть функция для проверки равенства массивов'),
], 8):
    print(corpus[2], "→", bow(corpus[2]))
    print(corpus[3], "→", bow(corpus[3]))
    print("Смысл противоположный, а векторы одинаковые: мешок слов не помнит порядок.")

<details><summary>💡 Подсказка</summary>

`np.array_equal(x, y)` возвращает True, если массивы совпадают.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
same = np.array_equal(___, ___)
```
</details>

---
# Часть 6. То же одной строкой: `CountVectorizer`

В библиотеке scikit-learn мешок слов уже написан:
* `fit` — выучить словарь;
* `transform` — посчитать слова;
* `fit_transform` — оба шага сразу.

### ✏️ Задание 9. Постройте мешок слов для `corpus` через `CountVectorizer`

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(token_pattern=r"[а-яё]+")   # наш же шаблон слов
X = None

In [ ]:
#@title 🧪 Проверка задания 9 { display-mode: "form" }
if check([
    ('словарь как у нас', lambda: list(cv.get_feature_names_out()) == vocab, 'Векторизатор должен сначала выучить словарь на corpus'),
    ('матрица как у нас', lambda: np.array_equal(X.toarray(), np.array([bow(t) for t in corpus])), 'Строка = твит, столбец = слово'),
], 9):
    print(cv.get_feature_names_out())
    print(X.toarray())
    print("Совпало с нашим bow() 🎉")

<details><summary>💡 Подсказка</summary>

Нужен метод, который и учит словарь, и сразу считает.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
X = cv.___(___)
```
</details>

### ✏️ Задание 10. Мешок слов для **всех** твитов: матрица `X_all`

In [ ]:
cv_all = CountVectorizer(token_pattern=r"[а-яё]+")
X_all = None

In [ ]:
#@title 🧪 Проверка задания 10 { display-mode: "form" }
if check([
    ('строк = твитов', lambda: X_all.shape[0] == 10000, 'Подайте столбец с текстами твитов'),
    ('столбцов = слов', lambda: X_all.shape[1] == V, 'Тот же шаблон слов — тот же словарь'),
], 10):
    zeros = 1 - X_all.nnz / (X_all.shape[0] * X_all.shape[1])
    print("Размер матрицы:", X_all.shape)
    print(f"Нулей в матрице: {zeros:.2%} — поэтому sklearn хранит её «сжато» (sparse)")

<details><summary>💡 Подсказка</summary>

Как в прошлом задании, только вместо `corpus` — столбец таблицы.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
X_all = cv_all.fit_transform(___)
```
</details>

### ✏️ Задание 11. Частоты слов в классах: `pos_counts` и `neg_counts`

In [ ]:
words = cv_all.get_feature_names_out()
is_pos = (df["label"] == "позитив").values

pos_counts = np.asarray(X_all[is_pos].sum(axis=0)).ravel()     # сумма по позитивным твитам
neg_counts = None

In [ ]:
#@title 🧪 Проверка задания 11 { display-mode: "form" }
if check([
    ('размер', lambda: len(neg_counts) == len(words), 'Должно быть по числу на каждое слово'),
    ('слов в негативе', lambda: neg_counts.sum() == X_all[~is_pos].sum(), 'Нужны НЕ позитивные строки: как перевернуть True/False в маске?'),
], 11):
    score = (pos_counts + 1) / (neg_counts + 1)
    freq = pos_counts + neg_counts >= 20
    top_pos = [words[i] for i in np.argsort(-score) if freq[i]][:15]
    top_neg = [words[i] for i in np.argsort(score) if freq[i]][:15]
    print("😊 чаще в позитиве:", ", ".join(top_pos))
    print("😞 чаще в негативе:", ", ".join(top_neg))

<details><summary>💡 Подсказка</summary>

Строка с `pos_counts` уже написана. Отличие одно — маска строк.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
neg_counts = np.asarray(X_all[___].sum(axis=0)).ravel()
```
</details>

---
# Часть 7. Чем плох мешок слов → TF-IDF

В мешке слов «и» весит столько же, сколько «кот». Но «и» есть **почти во всех** текстах — оно ничего не говорит о тексте. А «лает» есть только в одном — это важное слово.

**TF-IDF** = **TF** (часто ли слово в этом тексте) × **IDF** (редкое ли оно вообще).

Новый маленький корпус:

In [ ]:
docs = [
    "кот спит и кот ест",
    "пёс спит и пёс лает",
    "кот и пёс играют",
]
for d in docs:
    print(tokenize(d))

### TF — частота слова в тексте

$$TF(слово, текст) = \frac{\text{сколько раз слово в тексте}}{\text{сколько всего слов в тексте}}$$

«кот спит и кот ест»: слов 5, «кот» 2 раза → TF = 2/5 = 0.4.

### ✏️ Задание 12. Функция `tf(word, text)`

In [ ]:
def tf(word, text):
    words = tokenize(text)
    return None


In [ ]:
#@title 🧪 Проверка задания 12 { display-mode: "form" }
if check([
    ('кот в первом тексте', lambda: abs(tf('кот', docs[0]) - 0.4) < 1e-9, 'Должно получиться 2 / 5'),
    ('слова нет — 0', lambda: tf('лает', docs[0]) == 0, ''),
], 12):
    for w in ["кот", "и", "лает"]:
        print(f"TF({w!r}, {docs[0]!r}) = {tf(w, docs[0]):.2f}")

<details><summary>💡 Подсказка</summary>

У списка есть метод, который считает, сколько раз в нём встречается элемент.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
return words.___(word) / ___
```
</details>

### IDF — редкость слова

$$IDF(слово) = \log\frac{\text{сколько всего текстов}}{\text{в скольких текстах есть слово}}$$

* слово во **всех** текстах: log(3/3) = log 1 = **0** — слово бесполезно;
* слово в **одном** тексте: log(3/1) ≈ **1.1** — слово важное.

### ✏️ Задание 13. Функция `df_count(word)`: в скольких текстах `docs` есть слово

In [ ]:
def df_count(word):
    return None


In [ ]:
#@title 🧪 Проверка задания 13 { display-mode: "form" }
if check([
    ('«и» во всех трёх', lambda: df_count('и') == 3, 'Пройдите по всем текстам docs'),
    ('«кот» в двух', lambda: df_count('кот') == 2, 'Считаем тексты, а не повторы: «кот спит и кот ест» — это 1 текст'),
], 13):
    for w in ["и", "кот", "спит", "лает"]:
        print(f"{w!r} есть в {df_count(w)} текстах из {len(docs)}")

<details><summary>💡 Подсказка</summary>

`word in список` даёт True/False. Сумма True — это количество.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
return sum(___ for d in docs)
```
</details>

### ✏️ Задание 14. Функция `idf(word)`

In [ ]:
def idf(word):
    return None


In [ ]:
#@title 🧪 Проверка задания 14 { display-mode: "form" }
if check([
    ('«и» — 0', lambda: abs(idf('и')) < 1e-9, 'Слово во всех текстах должно получить 0'),
    ('«лает» — log 3', lambda: abs(idf('лает') - math.log(3)) < 1e-9, 'Натуральный логарифм — math.log'),
], 14):
    for w in ["и", "кот", "спит", "лает"]:
        print(f"IDF({w!r}) = {idf(w):.2f}")
    print("Чем реже слово, тем больше IDF.")

<details><summary>💡 Подсказка</summary>

Перепишите формулу: всего текстов — `len(docs)`, в скольких есть — ваша `df_count`.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
return math.log(___ / ___)
```
</details>

### ✏️ Задание 15. Функция `tfidf(word, text)` = TF × IDF

In [ ]:
def tfidf(word, text):
    return None


In [ ]:
#@title 🧪 Проверка задания 15 { display-mode: "form" }
if check([
    ('«и» обнулилось', lambda: tfidf('и', docs[0]) == 0, 'IDF слова «и» равен 0'),
    ('«кот» в первом тексте', lambda: abs(tfidf('кот', docs[0]) - 0.4 * math.log(1.5)) < 1e-9, 'Используйте обе свои функции'),
], 15):
    vocab3 = sorted({w for d in docs for w in tokenize(d)})
    table = pd.DataFrame([[round(tfidf(w, d), 2) for w in vocab3] for d in docs], columns=vocab3, index=docs)
    display(table)
    print("«и» — везде 0. «лает», «ест», «играют» — самые большие: они и отличают тексты.")

<details><summary>💡 Подсказка</summary>

Обе нужные функции уже написаны выше.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
return ___ * ___
```
</details>

### TF-IDF в sklearn

`TfidfVectorizer` делает то же самое, но с двумя поправками:
* к IDF прибавляет 1 (слово из всех текстов получает не 0, а немного);
* делит каждую строку на её длину, чтобы длинные и короткие твиты были сравнимы.

Цифры поэтому чуть другие, но **порядок** тот же: редкие слова важнее частых.

### ✏️ Задание 16. TF-IDF для `docs` через `TfidfVectorizer`: матрица `T`

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tv = TfidfVectorizer(token_pattern=r"[а-яё]+")
T = None

In [ ]:
#@title 🧪 Проверка задания 16 { display-mode: "form" }
if check([
    ('матрица 3 × 7', lambda: T.shape == (3, 7), 'Методы те же, что у CountVectorizer'),
    ('«и» — меньше всех в первом тексте', lambda: T[0, tv.vocabulary_['и']] < T[0, tv.vocabulary_['ест']], ''),
], 16):
    display(pd.DataFrame(T.toarray().round(2), columns=tv.get_feature_names_out(), index=docs))

<details><summary>💡 Подсказка</summary>

Всё как с `CountVectorizer`.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
T = tv.___(___)
```
</details>

---
# Часть 8. Учим модель угадывать настроение

Делим твиты: **80%** — учиться, **20%** — проверка (модель их не видит).

Модель — **логистическая регрессия**. Это один нейрон: у каждого слова есть **вес**, модель складывает веса слов твита. Сумма больше нуля — позитив, меньше — негатив.

⚠️ Векторизатор учим (`fit`) **только на обучающих** твитах. Для проверочных — только `transform`, иначе модель «подсмотрит» слова из проверки.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

train_text, test_text, y_train, y_test = train_test_split(df["text"], df["label"], test_size=0.2, random_state=0)
print("учимся на", len(train_text), "твитах, проверяем на", len(test_text))

### ✏️ Задание 17. Векторизуйте твиты мешком слов: `X_train`, `X_test`

In [ ]:
bow_vec = CountVectorizer(token_pattern=r"[а-яё]+")

X_train = None
X_test = None

In [ ]:
#@title 🧪 Проверка задания 17 { display-mode: "form" }
if check([
    ('обучающие', lambda: X_train.shape[0] == 8000, 'Словарь учим на train_text'),
    ('проверочные — тот же словарь', lambda: X_test.shape == (2000, X_train.shape[1]), 'Для проверки словарь не учим заново'),
], 17):
    print("X_train:", X_train.shape, " X_test:", X_test.shape)

<details><summary>💡 Подсказка</summary>

Для обучающих — метод, который учит и считает. Для проверочных — метод, который только считает.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
X_train = bow_vec.___(train_text)
X_test = bow_vec.___(test_text)
```
</details>

### ✏️ Задание 18. Обучите `model_bow` и посчитайте точность `acc_bow` на проверке

In [ ]:
model_bow = LogisticRegression(max_iter=1000)
# обучите модель
acc_bow = None

In [ ]:
#@title 🧪 Проверка задания 18 { display-mode: "form" }
if check([
    ('модель обучена', lambda: hasattr(model_bow, 'coef_'), 'У модели sklearn метод обучения называется так же, как у векторизатора'),
    ('точность лучше угадывания', lambda: acc_bow is not None and acc_bow > 0.6, 'Точность на проверке считает метод score'),
], 18):
    print(f"Точность (мешок слов): {acc_bow:.1%}   (наугад было бы 50%)")

<details><summary>💡 Подсказка</summary>

`fit(X, y)` — учиться, `score(X, y)` — доля правильных ответов.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
model_bow.fit(___, ___)
acc_bow = model_bow.score(___, ___)
```
</details>

### ✏️ Задание 19. То же с TF-IDF: `acc_tfidf`

In [ ]:
tfidf_vec = TfidfVectorizer(token_pattern=r"[а-яё]+")
Xt_train = None
Xt_test = None

model_tfidf = LogisticRegression(max_iter=1000)
# обучите модель
acc_tfidf = None

In [ ]:
#@title 🧪 Проверка задания 19 { display-mode: "form" }
if check([
    ('векторы', lambda: Xt_test.shape == (2000, Xt_train.shape[1]), 'Как в задании с мешком слов'),
    ('точность', lambda: acc_tfidf is not None and acc_tfidf > 0.6, 'Обучите и оцените'),
], 19):
    print(f"мешок слов: {acc_bow:.1%}   TF-IDF: {acc_tfidf:.1%}")

<details><summary>💡 Подсказка</summary>

Повторите два прошлых задания с `tfidf_vec` и `model_tfidf`.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
Xt_train = ___
Xt_test = ___
model_tfidf.fit(___, ___)
acc_tfidf = ___
```
</details>

> 🤔 Почему не 95%? Твиты короткие, а метки ставили по смайликам: «блин, опять дождь :)» — позитив. Даже человек не всегда угадает без смайлика. 65–70% на таких данных — честный результат.

---
# Часть 9. Что выучила модель
У каждого слова есть **вес**. Большой плюс — слово «тянет» к позитиву, большой минус — к негативу.

In [ ]:
words_t = tfidf_vec.get_feature_names_out()
w = model_tfidf.coef_[0]          # веса слов; плюс — к классу model_tfidf.classes_[1]
print("класс с плюсом:", model_tfidf.classes_[1])
order = np.argsort(w)
print("😞 тянут к негативу:", ", ".join(words_t[order[:15]]))
print("😊 тянут к позитиву:", ", ".join(words_t[order[-15:]][::-1]))

### ✏️ Задание 20. Напишите свой твит и узнайте, что скажет модель

In [ ]:
my_tweet = None   # ← любой текст в кавычках
answer = None

In [ ]:
#@title 🧪 Проверка задания 20 { display-mode: "form" }
if check([
    ('твит написан', lambda: isinstance(my_tweet, str) and len(tokenize(my_tweet)) > 0, 'Строка в кавычках, хотя бы одно русское слово'),
    ('модель ответила', lambda: answer in ('позитив', 'негатив'), 'Сначала текст в вектор, потом predict. Векторизатор ждёт список текстов'),
], 20):
    print(f"«{my_tweet}» → {answer}")
    print("Уверенность:", dict(zip(model_tfidf.classes_, model_tfidf.predict_proba(tfidf_vec.transform([my_tweet]))[0].round(2).tolist())))

<details><summary>💡 Подсказка</summary>

`transform` и `predict` работают со **списками**: заверните твит в `[...]`, а из ответа возьмите элемент `[0]`.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
answer = model_tfidf.___(tfidf_vec.___([___]))[0]
```
</details>

### ✏️ Задание 21. Проверьте «все плохо» и «все не плохо»: различает ли их модель?

In [ ]:
a1 = model_tfidf.predict(tfidf_vec.transform(["все плохо"]))[0]
a2 = None

In [ ]:
#@title 🧪 Проверка задания 21 { display-mode: "form" }
if check([
    ('оба ответа есть', lambda: a1 in ('позитив', 'негатив') and a2 in ('позитив', 'негатив'), 'Строка a1 — образец'),
], 21):
    print("«все плохо» →", a1, "| «все не плохо» →", a2)
    print("Модель видит отдельные слова. «не» и «плохо» по отдельности — оба негативные.")

<details><summary>💡 Подсказка</summary>

Посмотрите на строку с `a1`.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
a2 = ___
```
</details>

### 🔬 Эксперименты
* `TfidfVectorizer(token_pattern=r"[а-яё]+", ngram_range=(1, 2))` — словарь из слов **и пар слов** («не плохо» станет одним токеном). Точность выросла?
* `max_features=1000` — оставить только 1000 самых частых слов. Насколько упала точность?

### ✅ Итог
| | One-hot | Мешок слов | TF-IDF |
|---|---|---|---|
| что в векторе | одна 1 на месте слова | сколько раз слово в твите | частота × редкость |
| длина | весь словарь | весь словарь | весь словарь |
| частые слова («и», «я») | — | весят много | приглушены |
| порядок слов | — | теряется | теряется |
| смысл слов | не знает | не знает | не знает → **эмбеддинги** |